# RoBERTa-CRF-BoundaryAttention vs. ModernBERT-base — Fair Fold-2 Comparison

This notebook reproduces **the exact Fold 2 split** from your
`roberta_crf_boundary_attention_kfold.ipynb` notebook, trains a plain
**ModernBERT-base** token-classification baseline on that split, and
evaluates it against your already-trained **RoBERTa + CRF + Boundary
Attention** Fold 2 checkpoint (downloaded from
[`Souvikbasur/roberta-crf-news-ner-kfold-best`](https://huggingface.co/Souvikbasur/roberta-crf-news-ner-kfold-best)) —
**on the same held-out test sentences, with no retraining of the RoBERTa model.**

## ⚠️ Security note before you run this

Your uploaded notebook (cell with `os.environ["HF_TOKEN"] = "hf_jgrA..."`) has a **live Hugging Face
write token pasted in plaintext**. Anyone who sees that notebook (GitHub, a shared Colab link, this
chat) can use it to push/delete things in your HF account. Please:

1. Go to https://huggingface.co/settings/tokens and **revoke that token now**.
2. Create a new token and paste it only into Colab's **Secrets** tab (key icon in the left sidebar) as
   `HF_TOKEN`, or type it via the interactive prompt this notebook uses below — never hard-code it in a
   cell again.

This notebook does not reproduce that hard-coded token anywhere; it asks for a token interactively
(`getpass`) and only if the checkpoint repo needs auth (it looks public, so you may not even be
prompted).

## What "Fold 2" means here, and how it was generated

Your k-fold notebook does **not** do a random 80/10/10 split. It does a *sliding-window* 5-fold split:

1. `parse_label_studio(DATA_PATH)` loads every unique sentence from your Label Studio export into
   `tasks`, in file order, deduplicating on exact text match.
2. `build_folds(tasks, k=5, seed=42)` makes **one** seeded shuffle of `tasks`
   (`random.Random(42).shuffle`), then cuts the shuffled list into 5 contiguous, roughly-equal
   windows (`folds[0]`..`folds[4]`). The last fold absorbs the remainder if the dataset doesn't
   divide evenly by 5.
3. In the training loop, fold `i` (0-indexed) takes a turn as the **test** set:
   `test_tasks = folds[i]`. Everything else is pooled: `train_pool = folds[j] for j != i`
   (concatenated in fold order 0,1,2,3,4 skipping `i`).
4. `train_val_split_within_pool(train_pool, val_fraction=0.10, seed=42)` re-shuffles that pool
   with **the same seed 42** and slices off the first 10% as the validation set (for early
   stopping), the rest as train.

**Fold 2** (1-indexed, as printed by the original notebook's `FOLD {fold_idx+1}/{K_FOLDS}`) is
`fold_idx = 1`:

- **Test set**: `folds[1]` — the second contiguous window of the seed-42 shuffle. Never trained on
  by any fold-2 model.
- **Train pool**: `folds[0] + folds[2] + folds[3] + folds[4]`, concatenated in that order.
- **Train / Val split**: that pool, reshuffled with `Random(42)`, first 10% → val, remaining 90% →
  train.

Because steps 1–4 are deterministic given the same `DATA_PATH` file (same rows, same order) and the
same `SEED = 42`, re-running this exact code reproduces **byte-identical** train/val/test sentence
sets to the ones your original notebook used for fold 2 — **provided you upload the same
`17entity_news_ner_dataset.json` your original notebook used.** This notebook does not create a new
split, does not reshuffle differently, and does not change `SEED`, `K_FOLDS`, or `VAL_FRACTION_WITHIN_TRAIN`.

## Assumption about "the best-performing Fold 2 checkpoint"

Your original notebook's upload cell picks the checkpoint with `max(fold_results, key=lambda r:
r["seqeval_f1"])` — i.e. whichever fold scored highest, not necessarily fold index 1. You've told me
that in your run this happened to be Fold 2, so this notebook treats the checkpoint at
`Souvikbasur/roberta-crf-news-ner-kfold-best` as the fold-2 model and evaluates it on the fold-2 test
set reproduced above. If that checkpoint actually came from a different fold in your run, the
comparison below is no longer apples-to-apples — worth double-checking against your saved
`kfold_summary.csv` before trusting the final verdict.

## What changes vs. your notebook, and what doesn't

| Kept identical | Changed (minimum necessary) |
|---|---|
| Dataset, `ENTITY_TAGS`, BIO label scheme (`build_label_maps`) | ModernBERT backbone via `AutoModelForTokenClassification` instead of the custom RoBERTa-CRF-BoundaryAttention class |
| `SEED=42`, `K_FOLDS=5`, `VAL_FRACTION_WITHIN_TRAIN=0.10` | ModernBERT tokenizer has no `add_prefix_space` kwarg — that RoBERTa-only argument is dropped for it |
| `parse_label_studio`, `build_folds`, `train_val_split_within_pool`, `tokenize_and_align` (byte-for-byte) | ModernBERT has **no CRF, no boundary-attention loss, no custom `Trainer.compute_loss`/`prediction_step`** — plain cross-entropy token classification, per your instruction #7 |
| `LEARNING_RATE`, `BATCH_SIZE`, `GRAD_ACCUM_STEPS`, `NUM_EPOCHS`, `WARMUP_RATIO`, `WEIGHT_DECAY`, `EARLY_STOP_PAT` | seqeval `compute_metrics` needs an `argmax` step for ModernBERT (its `Trainer.predict` returns raw logits; the CRF model's custom trainer already returns decoded tag ids) |
| seqeval P/R/F1, nervaluate strict F1, the exact confusion-matrix/error-analysis taxonomy (`run_error_analysis`) | the error-analysis span-decoding function is split into two thin wrappers — one for CRF-decode output, one for argmax-logit output — that both feed the **same unchanged** matching/taxonomy body |


In [1]:
# Same installs as your notebook, plus a transformers version that supports ModernBERT
# (ModernBERT needs transformers>=4.48). If Colab's preinstalled torch/transformers are older,
# this upgrades them.
!pip install -q -U "transformers>=4.48.0" torch pytorch-crf seqeval nervaluate huggingface_hub safetensors pandas


KeyboardInterrupt: 

In [ ]:
pip install --upgrade transformers datasets pandas torch

In [ ]:
import torch
import torchvision
import transformers

print(torch.__version__)
print(torchvision.__version__)
print(transformers.__version__)

In [ ]:
!pip show torch torchvision transformers

In [6]:
!pip install -q \
pytorch-crf \
seqeval \
nervaluate \
huggingface_hub \
safetensors

In [7]:
import torch
import torchvision
import transformers

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)

Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
Transformers: 4.57.6


In [2]:
!pip uninstall -y transformers
!pip install -q "transformers>=4.48,<5"

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import transformers
print(transformers.__version__)

4.57.6


In [8]:
import json, random, os, csv, getpass
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from torchcrf import CRF
from seqeval.metrics import (
    classification_report, f1_score, precision_score, recall_score, accuracy_score,
)
from huggingface_hub import hf_hub_download

# ---- unchanged from your notebook ----
BASE_MODEL       = "FacebookAI/roberta-base"          # RoBERTa-CRF-BoundaryAttention backbone
DATA_PATH        = "17entity_news_ner_dataset.json"   # <-- upload the SAME file your original notebook used
OUTPUT_DIR       = "./ner_fold2_comparison_output"
MAX_LENGTH       = 256
SEED             = 42
K_FOLDS                   = 5
VAL_FRACTION_WITHIN_TRAIN = 0.10

LEARNING_RATE    = 1e-5
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01
EARLY_STOP_PAT   = 2

BOUNDARY_LOSS_WEIGHT = 0.3   # only used by the RoBERTa-CRF-BoundaryAttention class (for loading it)
ATTN_HEADS           = 4

device = "cuda" if torch.cuda.is_available() else "cpu"

ENTITY_TAGS = [
    "PER", "ORG", "LOC", "FAC", "DATE", "TIME", "EVT",
    "PROD", "MONEY", "PCT", "LAW", "NORP", "LANG",
    "WOA", "QTY", "ORD", "CARD",
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def build_label_maps():
    labels = ["O"]
    for tag in ENTITY_TAGS:
        labels.append(f"B-{tag}")
        labels.append(f"I-{tag}")
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label

label2id, id2label = build_label_maps()

# ---- new for this comparison ----
FOLD_NUMBER = 2                   # 1-indexed, matches the original notebook's printed "FOLD 2 / 5"
FOLD_IDX    = 2    # 0-indexed into `folds`

MODERNBERT_MODEL_NAME = "answerdotai/ModernBERT-base"   # baseline backbone, no CRF/extra heads
CRF_MODEL_REPO        = "Souvikbasur/roberta-crf-news-ner-kfold-best"  # your uploaded fold-2 checkpoint

print(f"Device: {device} | Total label classes: {len(label2id)} | K_FOLDS: {K_FOLDS} | "
      f"Reproducing Fold {FOLD_NUMBER} (fold_idx={FOLD_IDX})")


Device: cuda | Total label classes: 35 | K_FOLDS: 5 | Reproducing Fold 2 (fold_idx=2)


## 1. Load the dataset and build the same sliding-window folds

`parse_label_studio`, `build_folds`, `train_val_split_within_pool`, and `tokenize_and_align` below are
byte-for-byte copies of the functions in your k-fold notebook — same BIO-labeling fix (`B-` only fires
on the token whose `char_start` matches the entity's own start offset), same fold-construction logic.
`tokenize_and_align` is generic over any fast tokenizer that supports `return_offsets_mapping` and
`.word_ids()` — RoBERTa's and ModernBERT's tokenizers both qualify, so it is reused unchanged for both
models, per your instruction to keep tokenization logic the same unless ModernBERT specifically
requires a change (it doesn't, for this function).

In [23]:
def parse_label_studio(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    tasks, skipped, seen_texts = [], 0, set()
    for item in raw:
        text = item.get("data", {}).get("text", "").strip()
        if not text or text in seen_texts:
            skipped += 1
            continue
        seen_texts.add(text)
        entities = []
        annotations = item.get("annotations", [])
        if annotations:
            for result in annotations[0].get("result", []):
                val = result.get("value", {})
                start, end, labels = val.get("start"), val.get("end"), val.get("labels", [])
                if start is None or end is None or not labels:
                    continue
                if labels[0] in ENTITY_TAGS:
                    entities.append((start, end, labels[0]))
        tasks.append({"text": text, "entities": entities})
    print(f"[Parse] Loaded {len(tasks)} tasks, skipped {skipped}.")
    return tasks


def build_folds(tasks, k=K_FOLDS, seed=SEED):
    """Sliding-window k-fold split: shuffle once (seeded, reproducible), then
    cut the shuffled list into k contiguous, roughly-equal windows. Each
    window takes a turn as the held-out TEST fold while the rest become the
    train+val pool for that fold -- so across all k folds, every sentence is
    test-evaluated exactly once, by a model that never saw it in training."""
    shuffled = tasks[:]
    random.Random(seed).shuffle(shuffled)
    n = len(shuffled)
    fold_size = n // k
    folds, start = [], 0
    for i in range(k):
        end = start + fold_size if i < k - 1 else n   # last fold absorbs the remainder
        folds.append(shuffled[start:end])
        start = end
    sizes = [len(f) for f in folds]
    print(f"[Fold build] {k} folds from {n} sentences, sizes: {sizes}")
    assert sum(sizes) == n, "fold sizes must sum to the full dataset"
    return folds


def train_val_split_within_pool(pool, val_fraction=VAL_FRACTION_WITHIN_TRAIN, seed=SEED):
    p = pool[:]
    random.Random(seed).shuffle(p)
    n_val = int(len(p) * val_fraction)
    return p[n_val:], p[:n_val]    # train_tasks, val_tasks


def tokenize_and_align(task_list, tokenizer, label2id, max_length=MAX_LENGTH):
    """Fixed: token/entity matching now uses interval OVERLAP instead of strict full
    containment, so tokenizers whose offsets absorb a leading space or merge trailing
    punctuation/digits into a token no longer silently drop the whole gold entity.
    B- fires on the first token that overlaps the entity; I- on every subsequent one."""
    encoded = []
    for task in task_list:
        text = task["text"]
        entities = sorted(task["entities"], key=lambda x: x[0])
        tok = tokenizer(text, max_length=max_length, truncation=True,
                         padding="max_length", return_offsets_mapping=True, return_tensors=None)
        offsets = tok["offset_mapping"]
        word_ids = tok.word_ids()
        labels, b_start, b_end = [], [], []
        entity_started = [False] * len(entities)   # has this entity's B- fired yet?

        for offset, wid in zip(offsets, word_ids):
            if wid is None:
                labels.append(-100); b_start.append(-100); b_end.append(-100)
                continue

            char_start, char_end = offset
            if char_start == char_end:
                # zero-width offset (some special/added tokens) -- treat as O
                labels.append(label2id["O"]); b_start.append(0); b_end.append(0)
                continue

            matched_idx, matched_label, matched_end = None, "O", None
            for i, (ent_start, ent_end, ent_label) in enumerate(entities):
                # OVERLAP test (was: full containment) -- fixes the orphaned-entity bug
                if char_start < ent_end and ent_start < char_end:
                    matched_idx, matched_label, matched_end = i, ent_label, ent_end
                    break

            if matched_label == "O":
                bio = "O"
            elif not entity_started[matched_idx]:
                bio = f"B-{matched_label}"
                entity_started[matched_idx] = True
            else:
                bio = f"I-{matched_label}"
            labels.append(label2id.get(bio, label2id["O"]))

            is_start = 1 if bio.startswith("B-") else 0
            is_end = 1 if (matched_label != "O" and char_end >= matched_end) else 0
            b_start.append(is_start)
            b_end.append(is_end)

        encoded.append({
            "input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"],
            "labels": labels, "boundary_start": b_start, "boundary_end": b_end,
        })
    return encoded


class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        e = self.encodings[idx]
        return {
            "input_ids": torch.tensor(e["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(e["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(e["labels"], dtype=torch.long),
            "boundary_start": torch.tensor(e["boundary_start"], dtype=torch.long),
            "boundary_end": torch.tensor(e["boundary_end"], dtype=torch.long),
        }


set_seed(SEED)
tasks = parse_label_studio(DATA_PATH)
folds = build_folds(tasks, k=K_FOLDS, seed=SEED)


[Parse] Loaded 24044 tasks, skipped 822.
[Fold build] 5 folds from 24044 sentences, sizes: [4808, 4808, 4808, 4808, 4812]


## 2. Reproduce the exact Fold 2 split (no new split, no reshuffle)

This mirrors the body of the original k-fold loop for `fold_idx = FOLD_IDX` only — it does not run the
other 4 folds, and it does not touch `random`'s state differently than the original loop would have at
this point (each call below reseeds with the fixed `SEED`, exactly like `train_val_split_within_pool`
does internally).

In [24]:
test_tasks = folds[FOLD_IDX]
train_pool = [t for i, f in enumerate(folds) if i != FOLD_IDX for t in f]
train_tasks, val_tasks = train_val_split_within_pool(train_pool)

print(f"Fold {FOLD_NUMBER} reproduced:")
print(f"  Train: {len(train_tasks)} | Val (early-stopping monitor): {len(val_tasks)} | "
      f"Test (held-out window): {len(test_tasks)}")

# --- sanity checks: disjointness + full coverage of the (K-1)/K pool ---
train_texts = {t["text"] for t in train_tasks}
val_texts   = {t["text"] for t in val_tasks}
test_texts  = {t["text"] for t in test_tasks}
assert train_texts.isdisjoint(val_texts), "train/val overlap -- split is not reproducing correctly"
assert train_texts.isdisjoint(test_texts), "train/test overlap -- split is not reproducing correctly"
assert val_texts.isdisjoint(test_texts), "val/test overlap -- split is not reproducing correctly"
assert len(train_texts) + len(val_texts) == len(train_pool), "train+val must cover the whole pool"
print("Sanity check passed: train/val/test are disjoint and train+val fully covers the non-test pool.")


Fold 2 reproduced:
  Train: 17313 | Val (early-stopping monitor): 1923 | Test (held-out window): 4808
Sanity check passed: train/val/test are disjoint and train+val fully covers the non-test pool.


## 3. RoBERTa + CRF + Boundary Attention — model class (needed to load your checkpoint)

Unchanged from your notebook, verbatim (`BoundaryGuidedAttention`, `RobertaCRFBoundaryAttentionForNER`).
We need this class defined in this session purely to load the state dict you uploaded to
`Souvikbasur/roberta-crf-news-ner-kfold-best` — **we do not train this model**, only evaluate the
checkpoint you already trained.

In [25]:
class BoundaryGuidedAttention(nn.Module):
    """Multi-head self-attention where the attention logits get an additive
    bias from each KEY token's boundary evidence, so tokens carrying strong
    boundary signal pull more attention from every other token in the
    sequence -- not just their immediate neighbors. Padding keys are masked
    out entirely. Unchanged from the single-split notebook."""

    def __init__(self, hidden_size, num_heads=4):
        super().__init__()
        assert hidden_size % num_heads == 0, "hidden_size must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.bias_scale = nn.Parameter(torch.tensor(1.0))
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, H, boundary_score, attention_mask):
        batch, seq_len, hidden = H.shape

        def split_heads(x):
            return x.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.q_proj(H))
        K = split_heads(self.k_proj(H))
        V = split_heads(self.v_proj(H))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        key_bias = (self.bias_scale * boundary_score).unsqueeze(1).unsqueeze(1)
        scores = scores + key_bias

        key_mask = attention_mask.bool().unsqueeze(1).unsqueeze(1)
        scores = scores.masked_fill(~key_mask, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn)
        attn = self.dropout(attn)

        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, hidden)
        context = self.out_proj(context)

        return self.layer_norm(H + context)


class RobertaCRFBoundaryAttentionForNER(nn.Module):
    def __init__(self, base_model_name, num_labels, boundary_loss_weight=BOUNDARY_LOSS_WEIGHT,
                 attn_heads=ATTN_HEADS):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(base_model_name)
        hidden = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

        self.boundary_start_head = nn.Linear(hidden, 1)
        self.boundary_end_head = nn.Linear(hidden, 1)
        self.boundary_attention = BoundaryGuidedAttention(hidden, num_heads=attn_heads)

        self.boundary_loss_weight = boundary_loss_weight

    def forward(self, input_ids, attention_mask, labels=None,
                boundary_start=None, boundary_end=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        H = outputs.last_hidden_state

        start_logits = self.boundary_start_head(H).squeeze(-1)
        end_logits = self.boundary_end_head(H).squeeze(-1)
        boundary_score = torch.sigmoid(start_logits + end_logits)

        H_context = self.boundary_attention(H, boundary_score, attention_mask)

        emissions = self.classifier(self.dropout(H_context))
        mask = attention_mask.bool()

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_mask = mask & (labels != -100)
            crf_mask[:, 0] = True
            log_likelihood = self.crf(emissions, safe_labels, mask=crf_mask, reduction="mean")
            crf_loss = -log_likelihood

            valid = (boundary_start != -100).float()
            bs_target = (boundary_start == 1).float() * attention_mask.float()
            be_target = (boundary_end == 1).float() * attention_mask.float()
            bs_loss = F.binary_cross_entropy_with_logits(start_logits, bs_target, reduction="none")
            be_loss = F.binary_cross_entropy_with_logits(end_logits, be_target, reduction="none")
            bs_loss = (bs_loss * valid).sum() / valid.sum().clamp(min=1)
            be_loss = (be_loss * valid).sum() / valid.sum().clamp(min=1)
            boundary_loss = bs_loss + be_loss

            total_loss = crf_loss + self.boundary_loss_weight * boundary_loss
            return {"loss": total_loss, "crf_loss": crf_loss.detach(),
                    "boundary_loss": boundary_loss.detach(), "logits": emissions}
        else:
            decoded = self.crf.decode(emissions, mask=mask)
            return {"logits": emissions, "decoded": decoded,
                    "boundary_start_prob": torch.sigmoid(start_logits),
                    "boundary_end_prob": torch.sigmoid(end_logits)}


## 4. Shared evaluation helpers

`BoundaryAttentionTrainer` and `compute_metrics_fn` are unchanged from your notebook (needed for the
CRF model, which decodes through `model.crf.decode` rather than a plain argmax). `compute_metrics_fn_standard`
is the **one necessary addition**: ModernBERT's `Trainer.predict` returns raw logits
`(batch, seq_len, num_labels)`, so its predictions need an `argmax(-1)` before they line up with the
tag-id format `compute_metrics_fn` expects — the CRF model's custom `prediction_step` already returns
decoded tag ids, so it never needed this step.

`spans_overlap`, `classify_boundary`, and `run_error_analysis` are your exact matching/taxonomy logic,
unchanged. The only change is that `run_error_analysis` now takes a `predict_fn(text) -> spans`
argument instead of being hard-wired to the CRF model's `predict_spans_with_offsets` call, so the
identical taxonomy code can be reused for both models without duplicating it. Two thin `predict_fn`
wrappers below produce spans for each model's different output format (CRF decode vs. argmax logits).

In [37]:
class BoundaryAttentionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            labels=inputs["labels"], boundary_start=inputs["boundary_start"], boundary_end=inputs["boundary_end"],
        )
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        model.eval()
        with torch.no_grad():
            loss_out = model(
                input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                labels=inputs["labels"], boundary_start=inputs["boundary_start"], boundary_end=inputs["boundary_end"],
            )
            loss = loss_out["loss"].detach()
            decode_out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            decoded = decode_out["decoded"]
            max_len = inputs["labels"].shape[1]
            padded = torch.full((len(decoded), max_len), -100, dtype=torch.long)
            for i, seq in enumerate(decoded):
                padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        if prediction_loss_only:
            return (loss, None, None)
        return (loss, padded, inputs["labels"])


def compute_metrics_fn(id2label):
    """Unchanged -- expects preds already as decoded tag ids (CRF model's custom trainer)."""
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        true_list, pred_list = [], []
        for pred_seq, label_seq in zip(preds, labels):
            true_seq, pred_seq_ = [], []
            for p_id, l_id in zip(pred_seq, label_seq):
                if l_id == -100:
                    continue
                true_seq.append(id2label[int(l_id)])
                pred_seq_.append(id2label[int(p_id)] if int(p_id) != -100 else "O")
            true_list.append(true_seq)
            pred_list.append(pred_seq_)
        return {
            "precision": precision_score(true_list, pred_list),
            "recall": recall_score(true_list, pred_list),
            "f1": f1_score(true_list, pred_list),
        }
    return compute_metrics


def compute_metrics_fn_standard(id2label):
    """New (minimum necessary): same as compute_metrics_fn, but argmaxes raw logits first --
    needed because plain AutoModelForTokenClassification + the default Trainer returns logits,
    not decoded tag ids."""
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        preds = np.argmax(preds, axis=-1)
        true_list, pred_list = [], []
        for pred_seq, label_seq in zip(preds, labels):
            true_seq, pred_seq_ = [], []
            for p_id, l_id in zip(pred_seq, label_seq):
                if l_id == -100:
                    continue
                true_seq.append(id2label[int(l_id)])
                pred_seq_.append(id2label[int(p_id)])
            true_list.append(true_seq)
            pred_list.append(pred_seq_)
        return {
            "precision": precision_score(true_list, pred_list),
            "recall": recall_score(true_list, pred_list),
            "f1": f1_score(true_list, pred_list),
        }
    return compute_metrics


def spans_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def classify_boundary(g_start, g_end, p_start, p_end):
    if p_start <= g_start and p_end >= g_end:
        return "over_span"
    if p_start >= g_start and p_end <= g_end:
        return "under_span"
    return "shifted"


def _decode_tag_ids_to_spans(text, offsets, decoded, id2label_override=None):
    """Same as before, but trims leading/trailing whitespace off each reconstructed
    span using the source text -- fixes tokenizer-specific offset bleed (e.g. a
    leading space folded into a token's char_start) that was causing exact-match
    boundary comparisons to fail even when the tag sequence was correct."""
    _id2label = id2label_override if id2label_override is not None else id2label
    spans = []
    cur_start, cur_end, cur_type = None, None, None

    def _flush():
        s, e = cur_start, cur_end
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        if e > s:
            spans.append((s, e, cur_type))

    for (char_start, char_end), tag_id in zip(offsets, decoded):
        if char_start == char_end:
            continue
        tag = _id2label[tag_id]
        if tag == "O":
            if cur_type is not None:
                _flush()
                cur_type = None
            continue
        prefix, etype = tag.split("-", 1)
        if prefix == "B" or etype != cur_type:
            if cur_type is not None:
                _flush()
            cur_start, cur_end, cur_type = char_start, char_end, etype
        else:
            cur_end = char_end
    if cur_type is not None:
        _flush()
    return spans


def predict_spans_with_offsets_crf(text, tokenizer, model, max_length=MAX_LENGTH):
    enc = tokenizer(text, max_length=max_length, truncation=True,
                     return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    decoded = out["decoded"][0]
    return _decode_tag_ids_to_spans(text, offsets, decoded)


def predict_spans_with_offsets_argmax(text, tokenizer, model, id2label, max_length=MAX_LENGTH):
    enc = tokenizer(text, max_length=max_length, truncation=True,
                     return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    decoded = out.logits[0].argmax(dim=-1).tolist()
    return _decode_tag_ids_to_spans(text, offsets, decoded, id2label_override=id2label)

def run_error_analysis(test_tasks, predict_fn, sentence_idx_offset=0):
    """Same matching/taxonomy logic as your notebook's run_error_analysis, unchanged, except it now
    takes predict_fn(text) -> spans instead of (tokenizer, model) so it can drive either model."""
    confusion = defaultdict(lambda: defaultdict(int))
    error_rows = []

    for local_idx, task in enumerate(test_tasks):
        idx = sentence_idx_offset + local_idx
        text = task["text"]
        gold_spans = [(s, e, t) for s, e, t in task["entities"]]
        pred_spans = predict_fn(text)

        matched_pred, unmatched_gold = set(), []
        for g_start, g_end, g_type in gold_spans:
            match = None
            for p_idx, (p_start, p_end, p_type) in enumerate(pred_spans):
                if p_idx in matched_pred:
                    continue
                if p_start == g_start and p_end == g_end:
                    match = (p_idx, p_type)
                    break
            if match:
                p_idx, p_type = match
                matched_pred.add(p_idx)
                confusion[g_type][p_type] += 1
                if g_type != p_type:
                    error_rows.append({"sentence_idx": idx, "text": text[g_start:g_end], "context": text,
                                        "gold_type": g_type, "pred_type": p_type,
                                        "error_kind": "type_confusion", "boundary_kind": "exact"})
            else:
                unmatched_gold.append((g_start, g_end, g_type))

        unmatched_pred = [(p_idx, p_start, p_end, p_type)
                           for p_idx, (p_start, p_end, p_type) in enumerate(pred_spans)
                           if p_idx not in matched_pred]

        overlap_pairs = []
        for gi, (g_start, g_end, g_type) in enumerate(unmatched_gold):
            for p_idx, p_start, p_end, p_type in unmatched_pred:
                if spans_overlap(g_start, g_end, p_start, p_end):
                    ov_len = min(g_end, p_end) - max(g_start, p_start)
                    overlap_pairs.append((ov_len, gi, p_idx, g_start, g_end, g_type, p_start, p_end, p_type))
        overlap_pairs.sort(key=lambda x: -x[0])

        gold_boundary_matched, pred_boundary_matched = set(), set()
        for ov_len, gi, p_idx, g_start, g_end, g_type, p_start, p_end, p_type in overlap_pairs:
            if gi in gold_boundary_matched or p_idx in pred_boundary_matched:
                continue
            gold_boundary_matched.add(gi)
            pred_boundary_matched.add(p_idx)
            confusion[g_type]["MISSED"] += 1
            confusion["SPURIOUS"][p_type] += 1
            kind = classify_boundary(g_start, g_end, p_start, p_end)
            error_rows.append({"sentence_idx": idx, "text": text[g_start:g_end], "context": text,
                                "gold_type": g_type, "pred_type": p_type,
                                "error_kind": "boundary_error", "boundary_kind": kind})

        for gi, (g_start, g_end, g_type) in enumerate(unmatched_gold):
            if gi in gold_boundary_matched:
                continue
            is_fragment = any(spans_overlap(g_start, g_end, p_start, p_end) for _, p_start, p_end, _ in unmatched_pred)
            confusion[g_type]["MISSED"] += 1
            error_rows.append({"sentence_idx": idx, "text": text[g_start:g_end], "context": text,
                                "gold_type": g_type, "pred_type": "MISSED",
                                "error_kind": "true_miss" if not is_fragment else "boundary_error",
                                "boundary_kind": "fragmented" if is_fragment else ""})

        for p_idx, p_start, p_end, p_type in unmatched_pred:
            if p_idx in pred_boundary_matched:
                continue
            is_fragment = any(spans_overlap(p_start, p_end, g_start, g_end) for g_start, g_end, _ in unmatched_gold)
            confusion["SPURIOUS"][p_type] += 1
            error_rows.append({"sentence_idx": idx, "text": text[p_start:p_end], "context": text,
                                "gold_type": "SPURIOUS", "pred_type": p_type,
                                "error_kind": "true_spurious" if not is_fragment else "boundary_error",
                                "boundary_kind": "fragmented" if is_fragment else ""})

    gold_total_direct = sum(len(t["entities"]) for t in test_tasks)
    n_correct = sum(confusion[t].get(t, 0) for t in confusion if t != "SPURIOUS")
    n_type_confused = sum(1 for r in error_rows if r["error_kind"] == "type_confusion")
    n_boundary = sum(1 for r in error_rows if r["error_kind"] == "boundary_error" and r["gold_type"] != "SPURIOUS")
    n_true_miss = sum(1 for r in error_rows if r["error_kind"] == "true_miss")
    accounted = n_correct + n_type_confused + n_boundary + n_true_miss
    assert gold_total_direct == accounted, (
        f"MISMATCH in error analysis -- gold={gold_total_direct}, accounted={accounted}."
    )
    return confusion, error_rows


def confusion_to_dataframe(confusion):
    types = sorted({t for row in confusion.values() for t in row} | set(confusion.keys()))
    data = {p: [confusion[g].get(p, 0) for g in types] for p in types}
    return pd.DataFrame(data, index=types)


def error_taxonomy_summary(test_tasks, error_rows):
    gold_total = sum(len(t["entities"]) for t in test_tasks)
    n_type_confused = sum(1 for r in error_rows if r["error_kind"] == "type_confusion")
    n_boundary = sum(1 for r in error_rows if r["error_kind"] == "boundary_error" and r["gold_type"] != "SPURIOUS")
    n_true_miss = sum(1 for r in error_rows if r["error_kind"] == "true_miss")
    n_true_spurious = sum(1 for r in error_rows if r["error_kind"] == "true_spurious")
    n_correct = gold_total - n_type_confused - n_boundary - n_true_miss
    return {
        "gold_entities": gold_total,
        "correct": n_correct,
        "type_confusions": n_type_confused,
        "boundary_errors": n_boundary,
        "true_misses": n_true_miss,
        "true_spurious": n_true_spurious,
    }


## 5. ModernBERT-base tokenization on the reproduced Fold 2 split

**Minimum necessary tokenizer change:** `AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)`
in your notebook is a RoBERTa-tokenizer-specific keyword (needed only when the tokenizer receives
pre-split word lists). ModernBERT's tokenizer does not accept `add_prefix_space` and does not need it
here, since `tokenize_and_align` always calls the tokenizer on the raw sentence string, not a
pre-tokenized word list. Everything else about `tokenize_and_align` — the offset-based BIO assignment,
`max_length=256`, truncation, padding — is unchanged.

In [27]:
tokenizer_modernbert = AutoTokenizer.from_pretrained(MODERNBERT_MODEL_NAME)

train_ds_mb = NERDataset(tokenize_and_align(train_tasks, tokenizer_modernbert, label2id))
val_ds_mb   = NERDataset(tokenize_and_align(val_tasks, tokenizer_modernbert, label2id))
test_ds_mb  = NERDataset(tokenize_and_align(test_tasks, tokenizer_modernbert, label2id))

print(f"ModernBERT datasets built -- train={len(train_ds_mb)} val={len(val_ds_mb)} test={len(test_ds_mb)}")


ModernBERT datasets built -- train=17313 val=1923 test=4808


## 6. Train ModernBERT-base — plain token classification, no CRF / no extra heads

`AutoModelForTokenClassification.from_pretrained(...)` only, exactly per instruction #6/#7 — no
`BoundaryGuidedAttention`, no CRF layer, no boundary loss, no custom `Trainer` subclass. Training
hyperparameters (`LEARNING_RATE`, `BATCH_SIZE`, `GRAD_ACCUM_STEPS`, `NUM_EPOCHS`, `WARMUP_RATIO`,
`WEIGHT_DECAY`, early stopping patience) are identical to your k-fold loop's `TrainingArguments`. It
trains **only** on the reproduced Fold 2 `train_tasks`/`val_tasks` — not on the other 4 folds, and not
on the test set.

In [28]:
set_seed(SEED)

model_mb = AutoModelForTokenClassification.from_pretrained(
    MODERNBERT_MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
).to(device)

mb_output_dir = os.path.join(OUTPUT_DIR, "modernbert_fold2")
args_mb = TrainingArguments(
    output_dir=mb_output_dir,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to=[],
    save_total_limit=1,
)

trainer_mb = Trainer(
    model=model_mb,
    args=args_mb,
    train_dataset=train_ds_mb,
    eval_dataset=val_ds_mb,
    compute_metrics=compute_metrics_fn_standard(id2label),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)

trainer_mb.train()


Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.192900,0.190148,0.694212,0.774251,0.732051
2,0.147300,0.160798,0.751454,0.811010,0.780097
3,0.134400,0.151038,0.765695,0.824458,0.793991


TrainOutput(global_step=1626, training_loss=0.29579804964171125, metrics={'train_runtime': 1214.7948, 'train_samples_per_second': 42.755, 'train_steps_per_second': 1.338, 'total_flos': 8851335416133120.0, 'train_loss': 0.29579804964171125, 'epoch': 3.0})

## 7. Evaluate ModernBERT-base on the Fold 2 test set

In [29]:
preds_out_mb = trainer_mb.predict(test_ds_mb)
logits_mb, labels_mb = preds_out_mb.predictions, preds_out_mb.label_ids
preds_mb = np.argmax(logits_mb, axis=-1)

true_list_mb, pred_list_mb = [], []
for pred_seq, label_seq in zip(preds_mb, labels_mb):
    true_seq, pred_seq_ = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id == -100:
            continue
        true_seq.append(id2label[int(l_id)])
        pred_seq_.append(id2label[int(p_id)])
    true_list_mb.append(true_seq)
    pred_list_mb.append(pred_seq_)

mb_f1        = f1_score(true_list_mb, pred_list_mb)
mb_precision = precision_score(true_list_mb, pred_list_mb)
mb_recall    = recall_score(true_list_mb, pred_list_mb)
mb_accuracy  = accuracy_score(true_list_mb, pred_list_mb)

print(f"[ModernBERT-base | Fold {FOLD_NUMBER} test] "
      f"P={mb_precision:.4f} R={mb_recall:.4f} F1={mb_f1:.4f} token-acc={mb_accuracy:.4f}")
print()
print(classification_report(true_list_mb, pred_list_mb, digits=4))

from nervaluate import Evaluator
evaluator_mb = Evaluator(true_list_mb, pred_list_mb, tags=ENTITY_TAGS, loader="list")
sem_results_mb = evaluator_mb.evaluate()
if isinstance(sem_results_mb, tuple):
    sem_results_mb = sem_results_mb[0]
sem_overall_mb = sem_results_mb["overall"] if "overall" in sem_results_mb else sem_results_mb
strict_mb = sem_overall_mb["strict"]
mb_strict_f1 = strict_mb["f1"] if isinstance(strict_mb, dict) else strict_mb.f1
print(f"\nSemEval strict F1 (ModernBERT-base): {mb_strict_f1:.4f}")


[ModernBERT-base | Fold 2 test] P=0.7670 R=0.8223 F1=0.7937 token-acc=0.9563

              precision    recall  f1-score   support

        CARD     0.7990    0.8524    0.8248       928
        DATE     0.7883    0.8521    0.8189      1765
         EVT     0.3476    0.4219    0.3812       192
         FAC     0.2583    0.3679    0.3035       106
        LANG     0.8636    0.4634    0.6032        41
         LAW     0.4845    0.5493    0.5149       142
         LOC     0.8524    0.9070    0.8789      2516
       MONEY     0.7923    0.8545    0.8222       433
        NORP     0.8409    0.8595    0.8501      1039
         ORD     0.8294    0.8672    0.8479       241
         ORG     0.7010    0.7709    0.7343      2859
         PCT     0.8883    0.9244    0.9060       344
         PER     0.8596    0.8969    0.8779      2328
        PROD     0.4331    0.5062    0.4668       243
         QTY     0.7261    0.6734    0.6987       248
        TIME     0.6056    0.6301    0.6176       173
   

In [38]:
# --- confusion matrix + error-analysis taxonomy for ModernBERT, same taxonomy as your notebook ---
predict_fn_mb = lambda text: predict_spans_with_offsets_argmax(text, tokenizer_modernbert, model_mb, id2label)
confusion_mb, error_rows_mb = run_error_analysis(test_tasks, predict_fn_mb)

confusion_df_mb = confusion_to_dataframe(confusion_mb)
summary_mb = error_taxonomy_summary(test_tasks, error_rows_mb)

print("=== ModernBERT-base error taxonomy (Fold 2 test set) ===")
for k, v in summary_mb.items():
    print(f"  {k:16s}: {v}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
confusion_df_mb.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_modernbert_fold2.csv"))
pd.DataFrame(error_rows_mb).to_csv(os.path.join(OUTPUT_DIR, "error_log_modernbert_fold2.csv"), index=False)
print(f"\nWrote confusion_matrix_modernbert_fold2.csv and error_log_modernbert_fold2.csv to {OUTPUT_DIR}")
confusion_df_mb


=== ModernBERT-base error taxonomy (Fold 2 test set) ===
  gold_entities   : 13931
  correct         : 11374
  type_confusions : 479
  boundary_errors : 1754
  true_misses     : 324
  true_spurious   : 629

Wrote confusion_matrix_modernbert_fold2.csv and error_log_modernbert_fold2.csv to ./ner_fold2_comparison_output


,CARD,DATE,EVT,FAC,LANG,LAW,LOC,MISSED,MONEY,NORP,ORD,ORG,PCT,PER,PROD,QTY,SPURIOUS,TIME,WOA
CARD,791,11,2,0,0,0,0,112,3,0,1,0,0,0,2,5,0,2,0
DATE,6,1502,1,0,0,1,1,282,0,0,2,0,0,0,0,0,0,0,0
EVT,0,4,79,0,0,0,0,102,0,0,0,2,0,5,0,0,0,0,2
FAC,0,0,1,37,0,0,8,45,0,0,0,11,0,1,2,0,0,0,1
LANG,0,0,0,0,19,0,0,3,0,19,0,0,0,0,0,0,0,0,0
LAW,0,0,0,0,0,78,0,53,0,0,1,5,0,5,0,0,0,0,0
LOC,0,0,1,6,0,0,2286,192,0,3,0,25,0,10,0,0,0,0,5
MISSED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
MONEY,1,0,0,0,0,0,0,61,370,0,0,0,0,0,0,1,0,0,0
NORP,0,0,2,1,0,0,8,118,0,888,0,21,0,3,0,0,0,0,0


## 8. Download your RoBERTa + CRF + Boundary Attention Fold 2 checkpoint (no retraining)

Loads `Souvikbasur/roberta-crf-news-ner-kfold-best` and evaluates it, as-is, on the **same** Fold 2
`test_tasks` reproduced above. The tokenizer is instantiated the same way your training loop
instantiated it (`AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)`) rather than pulled
from the repo, because the tokenizer files your notebook's cell 30 uploaded were saved *without*
`add_prefix_space=True` — using the repo copy would silently diverge from the tokenizer that was
actually used to produce this checkpoint's labels during training.

In [31]:
# Only prompts for a token if the repo turns out to need auth -- most public HF repos don't.
hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    try:
        hf_hub_download(repo_id=CRF_MODEL_REPO, filename="label_map.json")
        print("Repo is public -- no token needed.")
    except Exception:
        hf_token = getpass.getpass("Enter your Hugging Face token (repo needs auth): ")

from safetensors.torch import load_file

ckpt_path  = hf_hub_download(repo_id=CRF_MODEL_REPO, filename="model.safetensors", token=hf_token)
label_path = hf_hub_download(repo_id=CRF_MODEL_REPO, filename="label_map.json", token=hf_token)

with open(label_path) as f:
    maps = json.load(f)
repo_label2id = maps["label2id"]
repo_id2label = {int(k): v for k, v in maps["id2label"].items()}

assert repo_label2id == label2id, (
    "Label map mismatch between the checkpoint repo and this notebook's ENTITY_TAGS-derived "
    "label2id -- the comparison below would not be apples-to-apples. Check ENTITY_TAGS matches "
    "what the checkpoint was trained with."
)
print("Label map from the checkpoint repo matches this notebook's label2id -- safe to proceed.")

tokenizer_roberta = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

crf_model = RobertaCRFBoundaryAttentionForNER(BASE_MODEL, num_labels=len(label2id)).to(device)
state_dict = load_file(ckpt_path)
missing, unexpected = crf_model.load_state_dict(state_dict, strict=False)
print("missing keys:", missing)
print("unexpected keys:", unexpected)   # both should be empty for a clean load
crf_model.eval()
print("RoBERTa-CRF-BoundaryAttention checkpoint loaded, no training performed.")


Repo is public -- no token needed.
Label map from the checkpoint repo matches this notebook's label2id -- safe to proceed.


Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


missing keys: []
unexpected keys: []
RoBERTa-CRF-BoundaryAttention checkpoint loaded, no training performed.


## 9. Evaluate the RoBERTa-CRF-BoundaryAttention checkpoint — same pipeline, same test set

Uses your unchanged `BoundaryAttentionTrainer` in **prediction-only mode** (`trainer.train()` is never
called) so the seqeval numbers go through the identical decode path (`model.crf.decode`) your k-fold
loop used for every fold's held-out window.

In [32]:
test_ds_roberta = NERDataset(tokenize_and_align(test_tasks, tokenizer_roberta, label2id))

eval_only_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "crf_fold2_eval_only"),
    per_device_eval_batch_size=BATCH_SIZE,
    report_to=[],
)

eval_trainer = BoundaryAttentionTrainer(model=crf_model, args=eval_only_args)
preds_out_crf = eval_trainer.predict(test_ds_roberta)
preds_crf, labels_crf = preds_out_crf.predictions, preds_out_crf.label_ids

true_list_crf, pred_list_crf = [], []
for pred_seq, label_seq in zip(preds_crf, labels_crf):
    true_seq, pred_seq_ = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id == -100:
            continue
        true_seq.append(id2label[int(l_id)])
        pred_seq_.append(id2label[int(p_id)] if int(p_id) != -100 else "O")
    true_list_crf.append(true_seq)
    pred_list_crf.append(pred_seq_)

crf_f1        = f1_score(true_list_crf, pred_list_crf)
crf_precision = precision_score(true_list_crf, pred_list_crf)
crf_recall    = recall_score(true_list_crf, pred_list_crf)
crf_accuracy  = accuracy_score(true_list_crf, pred_list_crf)

print(f"[RoBERTa-CRF-BoundaryAttention | Fold {FOLD_NUMBER} test] "
      f"P={crf_precision:.4f} R={crf_recall:.4f} F1={crf_f1:.4f} token-acc={crf_accuracy:.4f}")
print()
print(classification_report(true_list_crf, pred_list_crf, digits=4))

evaluator_crf = Evaluator(true_list_crf, pred_list_crf, tags=ENTITY_TAGS, loader="list")
sem_results_crf = evaluator_crf.evaluate()
if isinstance(sem_results_crf, tuple):
    sem_results_crf = sem_results_crf[0]
sem_overall_crf = sem_results_crf["overall"] if "overall" in sem_results_crf else sem_results_crf
strict_crf = sem_overall_crf["strict"]
crf_strict_f1 = strict_crf["f1"] if isinstance(strict_crf, dict) else strict_crf.f1
print(f"\nSemEval strict F1 (RoBERTa-CRF-BoundaryAttention): {crf_strict_f1:.4f}")


[RoBERTa-CRF-BoundaryAttention | Fold 2 test] P=0.9445 R=0.9617 F1=0.9530 token-acc=0.9900

              precision    recall  f1-score   support

        CARD     0.9288    0.9569    0.9426       927
        DATE     0.9437    0.9694    0.9564      1765
         EVT     0.9021    0.9115    0.9067       192
         FAC     0.8000    0.9057    0.8496       106
        LANG     0.9500    0.9268    0.9383        41
         LAW     0.8627    0.9296    0.8949       142
         LOC     0.9732    0.9794    0.9763      2520
       MONEY     0.9613    0.9746    0.9679       433
        NORP     0.9641    0.9817    0.9728      1039
         ORD     0.9008    0.9793    0.9384       241
         ORG     0.9383    0.9472    0.9427      2859
         PCT     0.9373    0.9564    0.9468       344
         PER     0.9637    0.9798    0.9717      2329
        PROD     0.9044    0.9342    0.9190       243
         QTY     0.9336    0.9073    0.9202       248
        TIME     0.8541    0.9133    0.8827

In [1]:
# --- confusion matrix + error-analysis taxonomy for the CRF model, identical taxonomy code ---
predict_fn_crf = lambda text: predict_spans_with_offsets_crf(text, tokenizer_roberta, crf_model)
confusion_crf, error_rows_crf = run_error_analysis(test_tasks, predict_fn_crf)

confusion_df_crf = confusion_to_dataframe(confusion_crf)
summary_crf = error_taxonomy_summary(test_tasks, error_rows_crf)

print("=== RoBERTa-CRF-BoundaryAttention error taxonomy (Fold 2 test set) ===")
for k, v in summary_crf.items():
    print(f"  {k:16s}: {v}")

confusion_df_crf.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_roberta_crf_fold2.csv"))
pd.DataFrame(error_rows_crf).to_csv(os.path.join(OUTPUT_DIR, "error_log_roberta_crf_fold2.csv"), index=False)
print(f"\nWrote confusion_matrix_roberta_crf_fold2.csv and error_log_roberta_crf_fold2.csv to {OUTPUT_DIR}")
confusion_df_crf


NameError: name 'run_error_analysis' is not defined

## 10. Final comparison — identical Fold 2 test set, identical evaluation pipeline

In [34]:
comparison = pd.DataFrame([
    {
        "Model": "ModernBERT-base (no CRF, standard baseline)",
        "Precision": mb_precision, "Recall": mb_recall, "F1": mb_f1,
        "SemEval strict F1": mb_strict_f1, "Token Accuracy": mb_accuracy,
        "n_test_sentences": len(test_tasks),
    },
    {
        "Model": "RoBERTa + CRF + Boundary Attention (Fold 2 checkpoint)",
        "Precision": crf_precision, "Recall": crf_recall, "F1": crf_f1,
        "SemEval strict F1": crf_strict_f1, "Token Accuracy": crf_accuracy,
        "n_test_sentences": len(test_tasks),
    },
]).set_index("Model")

comparison_path = os.path.join(OUTPUT_DIR, "fold2_model_comparison.csv")
comparison.to_csv(comparison_path)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print(f"Both models evaluated on the same {len(test_tasks)}-sentence Fold {FOLD_NUMBER} test set.\n")
display(comparison)

winner = comparison["F1"].idxmax()
f1_gap = abs(comparison.loc[comparison.index[0], "F1"] - comparison.loc[comparison.index[1], "F1"])
print(f"\n>>> Higher seqeval F1 on the Fold {FOLD_NUMBER} test set: {winner} "
      f"(by {f1_gap:.4f} F1 points)")
print(f"Wrote {comparison_path}")


Both models evaluated on the same 4808-sentence Fold 2 test set.



,Precision,Recall,F1,SemEval strict F1,Token Accuracy,n_test_sentences
Model,,,,,,
"ModernBERT-base (no CRF, standard baseline)",0.7670,0.8223,0.7937,0.7937,0.9563,4808
RoBERTa + CRF + Boundary Attention (Fold 2 checkpoint),0.9445,0.9617,0.9530,0.9530,0.9900,4808



>>> Higher seqeval F1 on the Fold 2 test set: RoBERTa + CRF + Boundary Attention (Fold 2 checkpoint) (by 0.1593 F1 points)
Wrote ./ner_fold2_comparison_output/fold2_model_comparison.csv


In [35]:
# --- side-by-side error-taxonomy comparison ---
taxonomy_compare = pd.DataFrame({
    "ModernBERT-base": summary_mb,
    "RoBERTa-CRF-BoundaryAttention": summary_crf,
})
print("Error taxonomy, side by side (counts over the same test entities):\n")
display(taxonomy_compare)


Error taxonomy, side by side (counts over the same test entities):



,ModernBERT-base,RoBERTa-CRF-BoundaryAttention
gold_entities,13931,13931
correct,950,13180
type_confusions,38,131
boundary_errors,12640,520
true_misses,303,100
true_spurious,628,268


## Caveats worth reading before trusting the verdict above

- **This is one fold, not the k-fold mean.** Your own notebook's section 9 makes the point that a
  single split/fold can be an outlier. This comparison is only as reliable as Fold 2 is representative
  of the other 4 — cross-check the `RoBERTa-CRF-BoundaryAttention` F1 above against the fold-2 row in
  your saved `kfold_summary.csv` (mean ± std across all 5 folds) before generalizing "which model wins"
  beyond this one fold.
- **The two models are not architecture-matched.** ModernBERT-base here is a *plain* token-classification
  head (linear layer + softmax + cross-entropy), per your instructions. The RoBERTa model carries a CRF
  sequence-decoding layer, a boundary-attention block, and an auxiliary boundary-detection loss on top
  of its backbone — plus it was **warm-started** from an already-trained checkpoint
  (`Souvikbasur/roberta-crf-news-ner`) rather than trained from the public pretrained RoBERTa weights.
  So this table answers *"does ModernBERT-base's plain classification head beat your full RoBERTa+CRF+
  BoundaryAttention system on Fold 2"* — it does not isolate whether the *gap* (if any) comes from the
  backbone (RoBERTa vs. ModernBERT) or from the extra architecture (CRF + boundary attention + warm
  start). A backbone-controlled comparison would need a plain-RoBERTa-baseline row too.
- **`ENTITY_TAGS` must match what the checkpoint was actually trained on.** The assertion in section 8
  checks this against the repo's `label_map.json` and will fail loudly if they don't line up — don't
  skip past that error if it fires.


In [36]:
# ==== Audit: does tokenize_and_align() lose different gold entities per tokenizer? ====
# Re-implements tokenize_and_align()'s label-assignment logic with counters, on the SAME
# test_tasks used by both models above. Doesn't change the original function or re-run
# training -- pure instrumentation. Run this after tokenizer_modernbert, tokenizer_roberta,
# and test_tasks already exist in the notebook's memory.

def diagnostic_tokenize_and_align(task_list, tokenizer, label2id, tokenizer_name, max_length=MAX_LENGTH):
    stats = {
        "tokenizer": tokenizer_name,
        "n_sentences": len(task_list),
        "n_sequences": 0,
        "total_labeled_tokens": 0,     # real tokens (wid is not None) -- includes "O"
        "total_ignored_tokens": 0,     # -100 (special/pad tokens)
        "n_truncated": 0,              # sentences whose untruncated length > max_length
        "n_entities_total": 0,
        "n_entities_orphaned": 0,      # entities with ZERO tokens satisfying the containment test
        "orphaned_by_type": defaultdict(int),
        "total_entities_by_type": defaultdict(int),
    }

    for task in task_list:
        text = task["text"]
        entities = sorted(task["entities"], key=lambda x: x[0])
        stats["n_entities_total"] += len(entities)
        for (_, _, etype) in entities:
            stats["total_entities_by_type"][etype] += 1

        # true length with no truncation, to detect how many sentences actually lose content
        untrunc = tokenizer(text, truncation=False, return_offsets_mapping=False)
        if len(untrunc["input_ids"]) > max_length:
            stats["n_truncated"] += 1

        tok = tokenizer(text, max_length=max_length, truncation=True,
                         padding="max_length", return_offsets_mapping=True, return_tensors=None)
        offsets = tok["offset_mapping"]
        word_ids = tok.word_ids()
        stats["n_sequences"] += 1

        entity_hit = [False] * len(entities)
        for offset, wid in zip(offsets, word_ids):
            if wid is None:
                stats["total_ignored_tokens"] += 1
                continue
            stats["total_labeled_tokens"] += 1
            char_start, char_end = offset
            for i, (ent_start, ent_end, ent_label) in enumerate(entities):
                if char_start >= ent_start and char_end <= ent_end and char_end > char_start:
                    entity_hit[i] = True
                    break

        for i, (ent_start, ent_end, etype) in enumerate(entities):
            if not entity_hit[i]:
                stats["n_entities_orphaned"] += 1
                stats["orphaned_by_type"][etype] += 1

    return stats


stats_mb  = diagnostic_tokenize_and_align(test_tasks, tokenizer_modernbert, label2id, "ModernBERT")
stats_crf = diagnostic_tokenize_and_align(test_tasks, tokenizer_roberta,    label2id, "RoBERTa-CRF")

print("=== Sequence-level stats ===")
seq_df = pd.DataFrame([
    {k: v for k, v in s.items() if not isinstance(v, defaultdict)}
    for s in (stats_mb, stats_crf)
]).set_index("tokenizer")
display(seq_df)

print("\n=== Per-type entity counts: total vs orphaned (never got a single labeled token) ===")
all_types = sorted(set(stats_mb["total_entities_by_type"]) | set(stats_crf["total_entities_by_type"]))
per_type_df = pd.DataFrame({
    "gold_total_mb":       [stats_mb["total_entities_by_type"].get(t, 0) for t in all_types],
    "orphaned_mb":         [stats_mb["orphaned_by_type"].get(t, 0) for t in all_types],
    "gold_total_roberta":  [stats_crf["total_entities_by_type"].get(t, 0) for t in all_types],
    "orphaned_roberta":    [stats_crf["orphaned_by_type"].get(t, 0) for t in all_types],
}, index=all_types)
per_type_df["surviving_mb"]      = per_type_df["gold_total_mb"] - per_type_df["orphaned_mb"]
per_type_df["surviving_roberta"] = per_type_df["gold_total_roberta"] - per_type_df["orphaned_roberta"]
display(per_type_df)

print("\n=== Sanity check ===")
print(f"Raw gold entities (tokenizer-independent, from test_tasks directly): "
      f"{sum(len(t['entities']) for t in test_tasks)}")
print(f"ModernBERT   -- sentences: {stats_mb['n_sentences']}, truncated: {stats_mb['n_truncated']}, "
      f"orphaned entities: {stats_mb['n_entities_orphaned']} / {stats_mb['n_entities_total']}")
print(f"RoBERTa-CRF  -- sentences: {stats_crf['n_sentences']}, truncated: {stats_crf['n_truncated']}, "
      f"orphaned entities: {stats_crf['n_entities_orphaned']} / {stats_crf['n_entities_total']}")

if stats_mb["n_entities_total"] != stats_crf["n_entities_total"]:
    raise AssertionError("test_tasks entity totals differ between the two runs -- this would be a "
                          "NEW and separate bug from the one under investigation.")

gap = stats_mb["n_entities_orphaned"] - stats_crf["n_entities_orphaned"]
print(f"\nOrphaned-entity gap (ModernBERT - RoBERTa): {gap}")
print("If this gap roughly matches the seqeval support gap you saw earlier (13,875 vs 9,195, "
      "i.e. ~4,680), the containment test at the `char_start >= ent_start and char_end <= ent_end` "
      "line in tokenize_and_align is the root cause, not truncation -- compare n_truncated between "
      "the two rows above to rule truncation in or out separately.")

=== Sequence-level stats ===


,n_sentences,n_sequences,total_labeled_tokens,total_ignored_tokens,n_truncated,n_entities_total,n_entities_orphaned
tokenizer,,,,,,,
ModernBERT,4808,4808,146810,1084038,0,13931,4736
RoBERTa-CRF,4808,4808,144477,1086371,0,13931,56



=== Per-type entity counts: total vs orphaned (never got a single labeled token) ===


,gold_total_mb,orphaned_mb,gold_total_roberta,orphaned_roberta,surviving_mb,surviving_roberta
CARD,929,541,929,2,388,927
DATE,1795,657,1795,30,1138,1765
EVT,194,21,194,2,173,192
FAC,106,7,106,0,99,106
LANG,41,29,41,0,12,41
LAW,142,5,142,0,137,142
LOC,2528,1303,2528,8,1225,2520
MONEY,433,12,433,0,421,433
NORP,1041,762,1041,2,279,1039
ORD,243,216,243,2,27,241



=== Sanity check ===
Raw gold entities (tokenizer-independent, from test_tasks directly): 13931
ModernBERT   -- sentences: 4808, truncated: 0, orphaned entities: 4736 / 13931
RoBERTa-CRF  -- sentences: 4808, truncated: 0, orphaned entities: 56 / 13931

Orphaned-entity gap (ModernBERT - RoBERTa): 4680
If this gap roughly matches the seqeval support gap you saw earlier (13,875 vs 9,195, i.e. ~4,680), the containment test at the `char_start >= ent_start and char_end <= ent_end` line in tokenize_and_align is the root cause, not truncation -- compare n_truncated between the two rows above to rule truncation in or out separately.
